# Quality Estimates
### Packages

In [127]:
import matplotlib
import numpy as np
import pandas as pd
import json
from fixedeffect.fe import fixedeffect

### Filter Countries

In [2]:
se_data = pd.read_csv('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/trade_and_the_se_2.csv')
se_data

,country,year,se_size,applied_tariff_rate,trade_tax,wa_rca,import_uvi,import_uvi_growth_rate,tax_burden,sales_tax,social_contributions,property_tax,income_tax,cpi,gov_eff,rule_law,reg_qua,cont_corr,size_agri
0,Albania,1995,39.18,NaN,2.330010,NaN,NaN,NaN,81.7,2.089194,3.457060,0.000000,1.488275,7.793219,NaN,NaN,NaN,NaN,32.837450
1,Albania,1996,37.07,NaN,2.305337,3.534493,NaN,NaN,81.7,2.714333,3.788204,0.000000,1.438076,12.725478,-0.688588,-0.684482,-0.474402,-0.893903,33.561661
2,Albania,1997,37.59,14.41,2.703781,3.054637,NaN,NaN,81.5,4.725193,3.690382,0.000000,1.084211,33.180274,NaN,NaN,NaN,NaN,28.941315
3,Albania,1998,38.16,NaN,3.277932,3.567234,NaN,NaN,82.1,7.475476,4.112596,0.000000,1.662960,20.642859,-0.732603,-0.929672,-0.441556,-0.992025,26.261043
4,Albania,1999,36.04,NaN,2.581083,4.370932,NaN,NaN,82.9,6.716613,4.095011,0.000000,2.335248,0.389438,NaN,NaN,NaN,NaN,23.622567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1486,Uruguay,2011,25.68,4.47,1.043245,4.485969,123.6,14.127424,84.3,8.209483,8.357692,1.030625,4.776377,8.092832,0.539624,0.731594,0.609196,1.289978,8.829162
1487,Uruguay,2012,23.25,4.20,1.027318,4.665257,123.0,-0.485437,81.2,8.105108,8.845312,1.039656,4.843578,8.097766,0.415859,0.614689,0.599089,1.378117,8.121772
1488,Uruguay,2013,22.49,4.69,1.012826,4.652748,119.8,-2.601626,84.2,7.918820,9.235893,1.026924,5.320543,8.575135,0.397322,0.576609,0.715801,1.397226,7.616201
1489,Uruguay,2014,20.59,4.68,1.044863,4.602556,114.7,-4.257095,77.4,7.761011,9.602884,1.080239,4.971522,8.877353,0.435041,0.761608,0.723711,1.388093,6.735462


### Fixing Country Names

In [3]:
with open('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Extracting PDF Data/country_correction.json') as json_file:
    country_corrections = json.load(json_file)['countries']

def fill_empty_correction_schemes():
    for cnt in country_corrections:
        if cnt['alternatives'] == [] and cnt['scheme'] == '':
            cnt['alternatives'] = [cnt['standard_name']]
            cnt['scheme'] = 'or_in'

fill_empty_correction_schemes()

def satisfies_scheme(country, scheme, alternatives):
    if scheme == 'exact':
        return country.lower() in alternatives
    elif scheme == 'all_in':
        return [alt in country.lower() for alt in alternatives] == [True] * len(alternatives)
    else:
        return [alt in country.lower() for alt in alternatives] != [False] * len(alternatives)

def correct_country(country):
    standard_names = [d['standard_name'] for d in country_corrections]
    if country.lower() in standard_names:
        return country.title()
    else:
        country_object = next((country_obj for country_obj in country_corrections if satisfies_scheme(country.lower(), country_obj['scheme'], country_obj['alternatives'])), '')
        return country_object['standard_name'].title() if country_object else ''
    
def get_accompanying_code(country):
    country_object = next((country_obj for country_obj in country_corrections if country.lower() == country_obj['standard_name']), '')
    return country_object['code'] if country_object else ''

with open('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Extracting PDF Data/exclusion_list.txt', 'r') as file:
    exclusion_lines = file.readlines()
    exclusion_lines = [line.rstrip('\n') for line in exclusion_lines]
exclusion_lines = exclusion_lines[1:]

In [4]:
def country_correct_dataset_1(df, column):
    df = df.rename(columns = {column: 'country'})

    df = df.query('country not in @exclusion_lines')
    df['country'] = df['country'].map(correct_country)
    # df['code'] = df['country'].map(get_accompanying_code)
    df = df.sort_values('country')
    df = df[df['country'] != '']
    return df

In [5]:
se_data['code'] = se_data['country'].map(get_accompanying_code)
all_countries = se_data['country'].unique()
all_country_codes = se_data['code'].unique()
all_country_codes

<StringArray>
['ALB', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BLR', 'BEL', 'BRA', 'CAN', 'CHL',
 'CHN', 'COL', 'CRI', 'HRV', 'CYP', 'DNK', 'DOM', 'ECU', 'SLV', 'EST', 'SWZ',
 'FJI', 'FIN', 'FRA', 'GEO', 'DEU', 'GRC', 'GTM', 'GUY', 'HUN', 'ISL', 'IND',
 'IDN', 'ISR', 'ITA', 'JPN', 'JOR', 'KAZ', 'LVA', 'LTU', 'LUX', 'MLT', 'MUS',
 'MEX', 'MDA', 'NPL', 'NLD', 'NZL', 'NOR', 'PER', 'POL', 'PRT', 'ROU', 'SEN',
 'SGP', 'SVN', 'ZAF', 'ESP', 'SUR', 'CHE', 'TJK', 'TZA', 'THA', 'TGO', 'TUR',
 'UKR', 'ARE', 'GBR', 'USA', 'URY']
Length: 71, dtype: str

## Testing A Single Product

In [6]:
horses_testing_init = pd.read_csv('TradeData_4_16_2026_13_30_51.csv', encoding='unicode_escape', index_col = False)
horses_testing = horses_testing_init[['period', 'cmdCode', 'reporterISO', 'reporterDesc', 'partnerISO', 'partnerDesc', 'cifvalue', 'qty']]
horses_testing = horses_testing[(horses_testing['partnerISO'] != 'W00') & (horses_testing['qty'] != 0)]
horses_testing = horses_testing.dropna(axis = 0)
horses_testing = horses_testing.rename(columns = {'period': 'year'})
print(horses_testing.columns)
horses_testing

Index(['year', 'cmdCode', 'reporterISO', 'reporterDesc', 'partnerISO',
       'partnerDesc', 'cifvalue', 'qty'],
      dtype='str')


,year,cmdCode,reporterISO,reporterDesc,partnerISO,partnerDesc,cifvalue,qty
1,2015,101,DZA,Algeria,BEL,Belgium,1029.000,1.0
2,2015,101,DZA,Algeria,FRA,France,943173.000,245.0
3,2015,101,DZA,Algeria,TUN,Tunisia,71260.000,8.0
8,2015,101,AGO,Angola,NAM,Namibia,7667.030,1.0
9,2015,101,AGO,Angola,PRT,Portugal,9009.770,4.0
...,...,...,...,...,...,...,...,...
1331,2015,101,URY,Uruguay,CHE,Switzerland,19125.000,2.0
1332,2015,101,URY,Uruguay,GBR,United Kingdom,44897.000,3.0
1333,2015,101,URY,Uruguay,USA,USA,98530.000,8.0
1337,2015,101,ZMB,Zambia,NAM,Namibia,941.506,2.0


In [ ]:
gravity_data_init = pd.read_csv('/Users/alex/Downloads/Gravity_csv_V202211/Gravity_V202211.csv')
gravity_data_init

In [24]:
gravity_data = gravity_data_init[gravity_data_init['year'] >= 1991]

In [25]:
gravity_data

,year,country_id_o,country_id_d,iso3_o,iso3_d,iso3num_o,iso3num_d,country_exists_o,country_exists_d,gmt_offset_2020_o,...,entry_time_o,entry_time_d,entry_tp_o,entry_tp_d,tradeflow_comtrade_o,tradeflow_comtrade_d,tradeflow_baci,manuf_tradeflow_baci,tradeflow_imf_o,tradeflow_imf_d
43,1991,ABW,ABW,ABW,ABW,533.0,533.0,1,1,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,1992,ABW,ABW,ABW,ABW,533.0,533.0,1,1,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,1993,ABW,ABW,ABW,ABW,533.0,533.0,1,1,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,1994,ABW,ABW,ABW,ABW,533.0,533.0,1,1,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,1995,ABW,ABW,ABW,ABW,533.0,533.0,1,1,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4699291,2017,ZWE,ZWE,ZWE,ZWE,716.0,716.0,1,1,2.0,...,61.0,61.0,70.0,70.0,NaN,NaN,NaN,NaN,NaN,NaN
4699292,2018,ZWE,ZWE,ZWE,ZWE,716.0,716.0,1,1,2.0,...,32.0,32.0,41.0,41.0,NaN,NaN,NaN,NaN,NaN,NaN
4699293,2019,ZWE,ZWE,ZWE,ZWE,716.0,716.0,1,1,2.0,...,27.0,27.0,36.0,36.0,NaN,NaN,NaN,NaN,NaN,NaN
4699294,2020,ZWE,ZWE,ZWE,ZWE,716.0,716.0,1,1,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
gravity_data.columns

Index(['year', 'country_id_o', 'country_id_d', 'iso3_o', 'iso3_d', 'iso3num_o',
       'iso3num_d', 'country_exists_o', 'country_exists_d',
       'gmt_offset_2020_o', 'gmt_offset_2020_d', 'distw_harmonic',
       'distw_arithmetic', 'distw_harmonic_jh', 'distw_arithmetic_jh', 'dist',
       'main_city_source_o', 'main_city_source_d', 'distcap', 'contig',
       'diplo_disagreement', 'scaled_sci_2021', 'comlang_off', 'comlang_ethno',
       'comcol', 'col45', 'legal_old_o', 'legal_old_d', 'legal_new_o',
       'legal_new_d', 'comleg_pretrans', 'comleg_posttrans',
       'transition_legalchange', 'comrelig', 'heg_o', 'heg_d', 'col_dep_ever',
       'col_dep', 'col_dep_end_year', 'col_dep_end_conflict', 'empire',
       'sibling_ever', 'sibling', 'sever_year', 'sib_conflict', 'pop_o',
       'pop_d', 'gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 'pop_source_o',
       'pop_source_d', 'gdp_source_o', 'gdp_source_d', 'gdp_ppp_o',
       'gdp_ppp_d', 'gdpcap_ppp_o', 'gdpcap_ppp_d', 'pop_pwt_o',

In [53]:
gravity_data_restricted = gravity_data[['iso3_o', 'iso3_d', 'year', 'contig', 'distw_arithmetic', 'comlang_off', 'comcol', 'col_dep_ever', 'col_dep', 'sibling_ever']]
gravity_data_restricted = gravity_data_restricted.rename(columns = {'iso3_o': 'reporterISO', 'iso3_d': 'partnerISO'})
gravity_data_restricted['partnerISO'].unique()
gravity_data_restricted = gravity_data_restricted.dropna()
gravity_data_restricted

,reporterISO,partnerISO,year,contig,distw_arithmetic,comlang_off,comcol,col_dep_ever,col_dep,sibling_ever
43,ABW,ABW,1991,0.0,32.0,0.0,0.0,0.0,0.0,0.0
44,ABW,ABW,1992,0.0,32.0,0.0,0.0,0.0,0.0,0.0
45,ABW,ABW,1993,0.0,32.0,0.0,0.0,0.0,0.0,0.0
46,ABW,ABW,1994,0.0,32.0,0.0,0.0,0.0,0.0,0.0
47,ABW,ABW,1995,0.0,32.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
4699291,ZWE,ZWE,2017,0.0,157.0,0.0,0.0,0.0,0.0,0.0
4699292,ZWE,ZWE,2018,0.0,157.0,0.0,0.0,0.0,0.0,0.0
4699293,ZWE,ZWE,2019,0.0,157.0,0.0,0.0,0.0,0.0,0.0
4699294,ZWE,ZWE,2020,0.0,154.0,0.0,0.0,0.0,0.0,0.0


In [54]:
testing = gravity_data_restricted[(gravity_data_restricted['reporterISO'] == 'AZE') & (gravity_data_restricted['year'] == 2015)]
testing[testing['partnerISO'] == 'DEU']

,reporterISO,partnerISO,year,contig,distw_arithmetic,comlang_off,comcol,col_dep_ever,col_dep,sibling_ever
284153,AZE,DEU,2015,0.0,3249.0,0.0,0.0,0.0,0.0,0.0


In [119]:
horses_testing_with_gravity = pd.merge(horses_testing, gravity_data_restricted, how = 'left', on = ['reporterISO', 'partnerISO', 'year'])
horses_testing_with_gravity = horses_testing_with_gravity.dropna()

horses_testing_with_gravity['unit_values'] = horses_testing_with_gravity['cifvalue'] / horses_testing_with_gravity['qty']
horses_testing_with_gravity

,year,cmdCode,reporterISO,reporterDesc,partnerISO,partnerDesc,cifvalue,qty,contig,distw_arithmetic,comlang_off,comcol,col_dep_ever,col_dep,sibling_ever,unit_values
0,2015,101,DZA,Algeria,BEL,Belgium,1029.000,1.0,0.0,1632.0,1.0,0.0,0.0,0.0,0.0,1029.000000
1,2015,101,DZA,Algeria,FRA,France,943173.000,245.0,0.0,1252.0,1.0,0.0,1.0,0.0,0.0,3849.685714
2,2015,101,DZA,Algeria,TUN,Tunisia,71260.000,8.0,1.0,636.0,1.0,1.0,0.0,0.0,1.0,8907.500000
3,2015,101,AGO,Angola,NAM,Namibia,7667.030,1.0,1.0,1479.0,0.0,0.0,0.0,0.0,0.0,7667.030000
4,2015,101,AGO,Angola,PRT,Portugal,9009.770,4.0,0.0,5922.0,1.0,0.0,1.0,0.0,0.0,2252.442500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,2015,101,URY,Uruguay,CHE,Switzerland,19125.000,2.0,0.0,11074.0,0.0,0.0,0.0,0.0,0.0,9562.500000
748,2015,101,URY,Uruguay,GBR,United Kingdom,44897.000,3.0,0.0,11046.0,0.0,0.0,0.0,0.0,0.0,14965.666667
749,2015,101,URY,Uruguay,USA,USA,98530.000,8.0,0.0,8941.0,0.0,0.0,0.0,0.0,1.0,12316.250000
750,2015,101,ZMB,Zambia,NAM,Namibia,941.506,2.0,1.0,1481.0,1.0,0.0,0.0,0.0,0.0,470.753000


In [120]:
gdp_per_cap_ppp = pd.read_csv('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - GDP per Capita PPP/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv')
gdp_per_cap_ppp = gdp_per_cap_ppp.drop(['Indicator Name', 'Indicator Code', 'Unnamed: 70'], axis = 1)
gdp_per_cap_ppp = gdp_per_cap_ppp.iloc[:,[0,1] + list(range(33, 63))]
gdp_per_cap_ppp = pd.melt(gdp_per_cap_ppp.reset_index(), id_vars = 'Country Code', value_vars = [str(yr) for yr in list(range(1991, 2020))], var_name = 'year')
gdp_per_cap_ppp['year'] = gdp_per_cap_ppp['year'].astype(int)
gdp_per_cap_ppp = gdp_per_cap_ppp.rename(columns = {'value': 'gdp_per_cap_ppp'})
gdp_per_cap_ppp

,Country Code,year,gdp_per_cap_ppp
0,ABW,1991,23099.940085
1,AFE,1991,1866.163505
2,AFG,1991,NaN
3,AFW,1991,2167.919776
4,AGO,1991,3740.350835
...,...,...,...
7709,XKX,2019,10861.053175
7710,YEM,2019,NaN
7711,ZAF,2019,13361.485129
7712,ZMB,2019,3361.397503


In [144]:
gdp_per_cap_ppp_reporter = gdp_per_cap_ppp.rename(columns = {'Country Code': 'reporterISO', 'gdp_per_cap_ppp': 'gdp_per_cap_ppp_reporter'})
gdp_per_cap_ppp_partner = gdp_per_cap_ppp.rename(columns = {'Country Code': 'partnerISO', 'gdp_per_cap_ppp': 'gdp_per_cap_ppp_partner'})
gdp_per_cap_ppp_reporter

horses_testing_with_gravity_w_gdp = pd.merge(horses_testing_with_gravity, gdp_per_cap_ppp_reporter, how = 'left', on = ['reporterISO', 'year'])
horses_testing_with_gravity_w_gdp = pd.merge(horses_testing_with_gravity_w_gdp, gdp_per_cap_ppp_partner, how = 'left', on = ['partnerISO', 'year'])
# horses_testing_with_gravity_w_reporter_gdp = horses_testing_with_gravity_w_reporter_gdp.drop(['value_x', 'value_y', 'value', 'gdp_per_cap_ppp_x', 'gdp_per_cap_ppp_y'])

# horses_testing_with_gravity = pd.merge(horses_testing_with_gravity, gdp_per_cap_ppp_partner, how = 'left', on = ['partnerISO', 'year'])
horses_testing_with_gravity_w_gdp = horses_testing_with_gravity_w_gdp.dropna()

In [145]:
horses_testing_with_gravity_w_gdp.describe()

,year,cmdCode,cifvalue,qty,contig,distw_arithmetic,comlang_off,comcol,col_dep_ever,col_dep,sibling_ever,unit_values,gdp_per_cap_ppp_reporter,gdp_per_cap_ppp_partner
count,734.0,734.0,7.340000e+02,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.0,734.000000,7.340000e+02,734.000000,734.000000
mean,2015.0,101.0,2.339403e+06,298.560733,0.148501,4431.275204,0.185286,0.036785,0.051771,0.0,0.173025,3.248314e+04,36220.069828,39437.678882
std,0.0,0.0,1.644626e+07,1580.739015,0.355839,4251.005673,0.388794,0.188361,0.221716,0.0,0.378526,1.125614e+05,18593.686215,19576.445541
min,2015.0,101.0,1.000000e+00,1.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.0,0.000000,4.535472e-01,910.393132,2317.279785
25%,2015.0,101.0,1.399100e+04,2.000000,0.000000,1094.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.920036e+03,22168.611419,23994.013672
50%,2015.0,101.0,7.260950e+04,10.000000,0.000000,2463.000000,0.000000,0.000000,0.000000,0.0,0.000000,6.794305e+03,37384.379646,40905.176775
75%,2015.0,101.0,4.568494e+05,47.750000,0.000000,7735.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.668732e+04,48772.069606,49201.064096
max,2015.0,101.0,3.553021e+08,22801.000000,1.000000,19043.000000,1.000000,1.000000,1.000000,0.0,1.000000,1.907644e+06,87156.187383,107859.685999


In [153]:
horses_testing_with_gravity_w_gdp['ln_cifvalue'] = np.log(horses_testing_with_gravity_w_gdp['cifvalue'])
horses_testing_with_gravity_w_gdp['ln_unit_values'] = np.log(horses_testing_with_gravity_w_gdp['unit_values'])
horses_testing_with_gravity_w_gdp['ln_reporter_income'] = np.log(horses_testing_with_gravity_w_gdp['gdp_per_cap_ppp_reporter'])
horses_testing_with_gravity_w_gdp['ln_partner_income'] = np.log(horses_testing_with_gravity_w_gdp['gdp_per_cap_ppp_partner'])
horses_testing_with_gravity_w_gdp['ln_distw_arithmetic'] = np.log(horses_testing_with_gravity_w_gdp['distw_arithmetic'])
horses_testing_with_gravity_w_gdp['ln_unit_values:ln_reporter_income'] = horses_testing_with_gravity_w_gdp['ln_unit_values'] * horses_testing_with_gravity_w_gdp['ln_reporter_income']
horses_testing_with_gravity_w_gdp['ln_reporter_income:ln_partner_income'] = horses_testing_with_gravity_w_gdp['ln_reporter_income'] * horses_testing_with_gravity_w_gdp['ln_partner_income']
horses_testing_with_gravity_w_gdp['ln_distw_arithmetic:ln_reporter_income'] = horses_testing_with_gravity_w_gdp['ln_distw_arithmetic'] * horses_testing_with_gravity_w_gdp['ln_reporter_income']

horses_testing_with_gravity_w_gdp = horses_testing_with_gravity_w_gdp.dropna()
horses_testing_with_gravity_w_gdp.to_csv('testing_gravity.csv')

y = ['ln_cifvalue']
exog = ['distw_arithmetic', 'comlang_off', 'comcol', 'col_dep_ever', 'col_dep', 'sibling_ever', 'contig']
category = ['reporterISO', 'partnerISO']
cluster = ['reporterISO', 'partnerISO']

formula = 'ln_cifvalue ~ distw_arithmetic + comlang_off + comcol + col_dep_ever + col_dep + sibling_ever + contig + ln_unit_values:ln_reporter_income + ln_reporter_income:ln_partner_income + ln_distw_arithmetic:ln_reporter_income|reporterISO + partnerISO|0'

model_fe = fixedeffect(data_df = horses_testing_with_gravity_w_gdp,
                       # dependent = y, exog_x = exog, category = category, cluster = cluster)
                      formula = formula)
result = model_fe.fit()
horses_testing_with_gravity_w_gdp.describe()
result.summary()

dependent variable(s): ['ln_cifvalue']
independent(exogenous): ['distw_arithmetic', 'comlang_off', 'comcol', 'col_dep_ever', 'col_dep', 'sibling_ever', 'contig', 'ln_unit_values:ln_reporter_income', 'ln_reporter_income:ln_partner_income', 'ln_distw_arithmetic:ln_reporter_income']
category variables(fixed effects): ['reporterISO', 'partnerISO']
cluster variables: ['0']


/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Quality Estimates/env/lib/python3.13/site-packages/fixedeffect/utils/DemeanDataframe.py:30: UserWarning: panel is unbalanced
  warnings.warn('panel is unbalanced')
/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Quality Estimates/env/lib/python3.13/site-packages/fixedeffect/utils/WaldTest.py:37: UserWarning: The variance matrix is either rank-deficient or indefinite.
  warnings.warn('The variance matrix is either rank-deficient or indefinite.')


KeyError: 0